# Nova 1 — training on a free Colab T4
Runtime → Change runtime type → **T4 GPU**. In the 🔑 Secrets panel add `HF_TOKEN` (a *write* token of your Hugging Face account) and enable notebook access. Then set the two names below and run all cells. When the session ends, open the notebook again and run all cells: training continues from the last checkpoint saved on Hugging Face.

In [ ]:
HF_USER = 'your-hf-username'          # change
DATA_REPO = f'{HF_USER}/nova1-data'    # made by scripts/push_to_hub.sh
CKPT_REPO = f'{HF_USER}/nova1-lora-ckpt'
MAX_MINUTES = 200                      # stop and save before Colab cuts the session

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%capture
!pip install -q unsloth
!pip install -q 'transformers>=5.0.0' 'trl>=0.24' 'peft>=0.17' datasets pyyaml huggingface_hub

In [ ]:
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
from huggingface_hub import snapshot_download
snapshot_download(DATA_REPO, repo_type='dataset', local_dir='/content/nova1')
%cd /content/nova1

In [ ]:
!python train/train.py --config train/config.colab.yaml --out out/nova1-lora --hub-repo $CKPT_REPO --max-minutes $MAX_MINUTES --resume

When the log says **saved LoRA adapter**, the stage is finished; the adapter is in `out/nova1-lora` and in the `last-checkpoint` folder of the checkpoint repo. GGUF export and evaluation run on the CPU box (`export/export_gguf.sh`, `eval/run_eval.py`).